<a href="https://colab.research.google.com/github/IshiPareek/mech_interpet/blob/main/02_gpt2_ablation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install transformer_lens
import transformer_lens
from transformer_lens import HookedTransformer, utils
from transformer_lens.hook_points import HookPoint
import torch
import numpy as np
from jaxtyping import Float, Int
import einops

print(f"TransformerLens installd")
print(f"CUDA available: {torch.cuda.is_available()}")

In [ ]:
model = HookedTransformer.from_pretrained("gpt2")

In [ ]:
tokens = model.to_tokens("The meaning of life is")
str_tokens = model.to_str_tokens(tokens)
print(model.to_str_tokens(tokens))
logits = model(tokens)
print(logits.shape)

In [ ]:
# ── VISUALIZATION BOILERPLATE ── define once, call anywhere ──────────────────

import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np

def viz(cache, plot_type, layer=0, head=0, dims=64, model=model, str_tokens=str_tokens, original_loss=None, ablated_loss=None):
    """
    Universal cache visualizer.

    plot_type options:
        "embed"      → token embeddings
        "pos_embed"  → positional embeddings
        "pattern"    → attention pattern for one head
        "all_heads"  → all 12 heads in a layer
        "resid"      → residual stream at a layer
        "logit_lens" → top predicted token across all layers
        "qkv"        → query, key, value vectors
    """

    def heatmap(ax, matrix, title, row_labels=None, cmap="RdBu_r"):
        vmax = np.abs(matrix).max()
        ax.imshow(matrix, cmap=cmap, vmin=-vmax, vmax=vmax, aspect="auto")
        ax.set_title(title, fontsize=10, pad=6)
        if row_labels:
            ax.set_yticks(range(len(row_labels)))
            ax.set_yticklabels(row_labels, fontsize=8)
        else:
            ax.set_yticks([])
        ax.set_xticks([])

    # ── embed ────────────────────────────────────────────────────────────────
    if plot_type == "embed":
        data = cache["hook_embed"][0, :, :dims].detach().numpy()
        fig, ax = plt.subplots(figsize=(14, 3))
        heatmap(ax, data, f"hook_embed — first {dims} dims", row_labels=str_tokens)
        plt.tight_layout(); plt.show()

    # ── pos_embed ────────────────────────────────────────────────────────────
    elif plot_type == "pos_embed":
        data = cache["hook_pos_embed"][0, :, :dims].detach().numpy()
        fig, ax = plt.subplots(figsize=(14, 3))
        heatmap(ax, data, f"hook_pos_embed — first {dims} dims", row_labels=str_tokens, cmap="PiYG")
        plt.tight_layout(); plt.show()

    # ── pattern ──────────────────────────────────────────────────────────────
    elif plot_type == "pattern":
        data = cache[f"blocks.{layer}.attn.hook_pattern"][0, head].detach().numpy()
        fig, ax = plt.subplots(figsize=(5, 4))
        ax.imshow(data, cmap="Blues", vmin=0, vmax=1)
        ax.set_xticks(range(len(str_tokens))); ax.set_xticklabels(str_tokens, rotation=45, ha="right", fontsize=8)
        ax.set_yticks(range(len(str_tokens))); ax.set_yticklabels(str_tokens, fontsize=8)
        ax.set_title(f"attention pattern — layer {layer}, head {head}", fontsize=10)
        plt.tight_layout(); plt.show()

    # ── all_heads ────────────────────────────────────────────────────────────
    elif plot_type == "all_heads":
        fig, axes = plt.subplots(3, 4, figsize=(14, 9))
        for h, ax in enumerate(axes.flat):
            data = cache[f"blocks.{layer}.attn.hook_pattern"][0, h].detach().numpy()
            ax.imshow(data, cmap="Blues", vmin=0, vmax=1)
            ax.set_title(f"H{h}", fontsize=9)
            ax.set_xticks([]); ax.set_yticks([])
        fig.suptitle(f"all attention heads — layer {layer}", fontsize=12)
        plt.tight_layout(); plt.show()

    # ── resid ────────────────────────────────────────────────────────────────
    elif plot_type == "resid":
        data = cache[f"blocks.{layer}.hook_resid_post"][0, :, :dims].detach().numpy()
        fig, ax = plt.subplots(figsize=(14, 3))
        heatmap(ax, data, f"hook_resid_post — layer {layer}, first {dims} dims", row_labels=str_tokens)
        plt.tight_layout(); plt.show()

    # ── logit_lens ───────────────────────────────────────────────────────────
    elif plot_type == "logit_lens":
        n_layers = model.cfg.n_layers
        cell_text = []
        for l in range(n_layers):
            resid = cache[f"blocks.{l}.hook_resid_post"]
            top = model.unembed(model.ln_final(resid))[0, :, :].argmax(dim=-1)
            cell_text.append([model.to_string(t) for t in top])
        fig, ax = plt.subplots(figsize=(14, 5))
        ax.axis("off")
        tbl = ax.table(cellText=cell_text, rowLabels=[f"L{l}" for l in range(n_layers)],
                       colLabels=str_tokens, loc="center", cellLoc="center")
        tbl.auto_set_font_size(False); tbl.set_fontsize(8); tbl.scale(1, 1.3)
        ax.set_title("logit lens — top predicted token per layer × position", fontsize=11, pad=20)
        plt.tight_layout(); plt.show()

    # ── qkv ──────────────────────────────────────────────────────────────────
    elif plot_type == "qkv":
        fig, axes = plt.subplots(1, 3, figsize=(14, 3))
        for ax, key, label in zip(axes, ["hook_q","hook_k","hook_v"], ["Query","Key","Value"]):
            data = cache[f"blocks.{layer}.attn.{key}"][0, :, head, :].detach().numpy()
            heatmap(ax, data, f"{label} — layer {layer}, head {head}", row_labels=str_tokens)
        plt.tight_layout(); plt.show()

## ABLATION
    elif plot_type == "ablation":
      if original_loss is None or ablated_loss is None :
        print ("pass original_loss and ablation_loss to use this plot type")
        return

      orig = original_loss.item()
      abla = ablated_loss.item()
      diff = abla - orig

      fig, ax = plt.subplots(figsize=(5, 4))
      bars = ax.bar(
      ["Original Loss", "Ablated Loss"],
      [orig, abla],
      color=["steelblue", "tomato"])

      ax.bar_label(bars, fmt="%.3f", padding=4, fontsize=10)
      ax.set_title(
            f"Head Ablation — Layer {layer}, Head {head}\nΔ Loss = +{diff:.3f}",
      fontsize=11)
      ax.set_ylabel("Cross Entropy Loss")
      ax.set_ylim(0, max(orig, abla) * 1.2)
      plt.tight_layout(); plt.show()

    else:
        print(f"unknown plot_type '{plot_type}'. options: embed, pos_embed, pattern, all_heads, resid, logit_lens, qkv")

In [ ]:
layer_to_ablate = 0
head_to_ablate = 8

# ── hook function (only does one thing) ──────────────────────────
def head_ablation_hook(
    value: Float[torch.Tensor, "batch pos head_index d_head"], hook: HookPoint
) -> Float[torch.Tensor, "batch pos head_index d_head"]:
    print(f"Shape of the value tensor : {value.shape}")
    value[:, :, head_to_ablate, :] = 0.0
    return value

# ── run and compare (outside the hook) ───────────────────────────
original_loss = model(tokens, return_type="loss")
ablated_loss = model.run_with_hooks(tokens,
    return_type="loss",
    fwd_hooks=[(utils.get_act_name("v", layer_to_ablate), head_ablation_hook)],
)

print(f"Original Loss : {original_loss.item():.3f}")
print(f"Ablated Loss  : {ablated_loss.item():.3f}")

# ── cache and visualize ───────────────────────────────────────────
logits, cache = model.run_with_cache(tokens)
str_tokens = model.to_str_tokens(tokens)
viz(cache, "ablation",
    layer=layer_to_ablate,
    head=head_to_ablate,
    original_loss=original_loss,
    ablated_loss=ablated_loss)

In [ ]:
deltas = []

for layer_idx in range(model.cfg.n_layers):
    def make_hook(h):
        def head_ablation_hook(
            value: Float[torch.Tensor, "batch pos head_index d_head"], hook: HookPoint
        ) -> Float[torch.Tensor, "batch pos head_index d_head"]:
            value[:, :, h, :] = 0.0
            return value
        return head_ablation_hook

    ablated = model.run_with_hooks(
        tokens,
        return_type="loss",
        fwd_hooks=[(utils.get_act_name("v", layer_idx), make_hook(head_to_ablate))],
    )
    deltas.append(ablated.item() - original_loss.item())

# plot
fig, ax = plt.subplots(figsize=(10, 4))
bars = ax.bar(range(model.cfg.n_layers), deltas, color=["tomato" if d > 0 else "steelblue" for d in deltas])
ax.bar_label(bars, fmt="%.3f", padding=4, fontsize=8)
ax.set_xlabel("Layer")
ax.set_ylabel("Δ Loss (ablated - original)")
ax.set_title(f"Head {head_to_ablate} ablation across all layers\nhigher = more important")
ax.set_xticks(range(model.cfg.n_layers))
ax.axhline(0, color="black", linewidth=0.8)
plt.tight_layout()
plt.show()

In [ ]:
layer_to_ablate = 0
head_index_to_ablate = 8

n_layers = model.cfg.n_layers
n_heads = model.cfg.n_heads
original_loss = model(tokens, return_type="loss")

# matrix to store all deltas (n_layers x n_heads)
delta_matrix = np.zeros((n_layers, n_heads))

for layer_idx in range(n_layers):
    for head_idx in range(n_heads):
        def make_hook(h):
            def head_ablation_hook(
                value: Float[torch.Tensor, "batch pos head_index d_head"], hook: HookPoint
            ) -> Float[torch.Tensor, "batch pos head_index d_head"]:
                value[:, :, h, :] = 0.0
                return value
            return head_ablation_hook

        ablated_loss = model.run_with_hooks(
            tokens,
            return_type="loss",
            fwd_hooks=[(utils.get_act_name("v", layer_idx), make_hook(head_idx))],
        )
        delta_matrix[layer_idx, head_idx] = ablated_loss.item() - original_loss.item()

# plot
fig, ax = plt.subplots(figsize=(14, 6))
vmax = np.abs(delta_matrix).max()
im = ax.imshow(delta_matrix, cmap="RdBu_r", vmin=-vmax, vmax=vmax, aspect="auto")
plt.colorbar(im, ax=ax, label="Δ Loss (ablated - original)")
ax.set_xlabel("Head")
ax.set_ylabel("Layer")
ax.set_title("Loss increase from ablating each head\nred = important, blue = suppressive")
ax.set_xticks(range(n_heads))
ax.set_yticks(range(n_layers))
ax.set_xticklabels([f"H{h}" for h in range(n_heads)])
ax.set_yticklabels([f"L{l}" for l in range(n_layers)])

# annotate each cell with the delta value
for i in range(n_layers):
    for j in range(n_heads):
        ax.text(j, i, f"{delta_matrix[i,j]:.2f}", ha="center", va="center", fontsize=7)

plt.tight_layout()
plt.show()